# 02 SFT Review

Purpose: launch the Stage 1 QLoRA SFT run from the Drive-backed source tree, inspect the saved summary and curves, and review a few tuned sample outputs against the held-out eval rows.

Precondition: run `00_colab_setup.ipynb` first.


In [ ]:
from pathlib import Path

SOURCE_ROOT = Path('/content/drive/MyDrive/json-ft-source')
RUNTIME_ROOT = Path('/content/drive/MyDrive/json-ft-runs')
RUN_NAME = 'sft-colab-run'

SOURCE_ROOT, RUNTIME_ROOT


In [ ]:
!python /content/drive/MyDrive/json-ft-source/scripts/train_sft.py \
    --config /content/drive/MyDrive/json-ft-source/configs/sft.yaml \
    --profile dev \
    --run-name {RUN_NAME} \
    --runtime-root /content/drive/MyDrive/json-ft-runs \
    --mirror-metrics-to-repo \
    --mirror-plots-to-repo \
    --mirror-checkpoint-manifest-to-repo


In [ ]:
import json

summary_path = RUNTIME_ROOT / 'persistent' / 'metrics' / f'{RUN_NAME}_sft_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary


In [ ]:
from IPython.display import Image, display

for key in ('loss_curve_path', 'eval_loss_curve_path'):
    curve_path = Path(summary.get(key, ''))
    if curve_path.exists():
        print(f'{key}: {curve_path}')
        display(Image(filename=str(curve_path)))
    else:
        print(f'{key}: not generated for this run')


In [ ]:
import json
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

eval_manifest_path = SOURCE_ROOT / 'data' / 'manifests' / 'support_tickets_eval_manifest.jsonl'
eval_rows = [json.loads(line) for line in eval_manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
tokenizer = AutoTokenizer.from_pretrained(summary['base_model'], trust_remote_code=False)
tokenizer.eos_token = '<|im_end|>'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    summary['base_model'],
    quantization_config=quantization_config,
    torch_dtype=compute_dtype,
    device_map='auto',
    trust_remote_code=False,
)
model = PeftModel.from_pretrained(base_model, summary['adapter_path'])
model.eval()

device = 'cuda' if torch.cuda.is_available() else 'cpu'
for row in eval_rows[:3]:
    prompt_messages = row['messages'][:-1]
    encoded = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    )
    encoded = encoded.to(device)
    generated = model.generate(
        encoded,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prediction_tokens = generated[0][encoded.shape[-1]:]
    prediction = tokenizer.decode(prediction_tokens, skip_special_tokens=True).strip()
    print(f"\n=== {row['record_id']} ===")
    print('PROMPT:')
    print(prompt_messages)
    print('\nTUNED OUTPUT:')
    print(prediction)
    print('\nREFERENCE JSON:')
    print(row['reference_json'])
